In [1]:
import os
import numpy as np
import cv2
import nibabel as nib
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# --- CONFIGURATION ---
IMG_SIZE = 224 # EfficientNetB0 default
BATCH_SIZE = 16
base_path = "/kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas"
image_path = os.path.join(base_path, "imagesTr")
mask_path = os.path.join(base_path, "labelsTr")

# 1. CLEAN FILE LOADING
all_files = sorted([f for f in os.listdir(image_path) 
                    if (f.endswith(".nii") or f.endswith(".nii.gz")) 
                    and not f.startswith(".")])

train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)

# --- FOCAL LOSS FOR SMALL TUMOR SENSITIVITY ---
# Gamma=2.0 focuses on hard-to-classify small tumors
# Alpha=0.25 balances the class importance
def focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        
        # Calculate Focal Loss components
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        fl = - alpha_t * tf.pow(1.0 - p_t, gamma) * tf.math.log(p_t)
        return tf.reduce_mean(fl)
    return focal_loss_fixed

# --- BALANCED GENERATOR ---
class PancreasGenerator(Sequence):
    def __init__(self, patient_files, is_training=True):
        self.patient_files = patient_files
        self.is_training = is_training
        self.samples = self._prepare_samples()
        self.cache = {}

    def _prepare_samples(self):
        tumor_samples = []
        healthy_samples = []
        for file in self.patient_files:
            m_path = os.path.join(mask_path, file)
            mask_vol = nib.load(m_path).get_fdata()
            for i in range(mask_vol.shape[2]):
                if np.any(mask_vol[:, :, i] == 2):
                    tumor_samples.append((file, i, 1))
                elif self.is_training:
                    # Randomly downsample healthy slices during training
                    if np.random.random() < 0.12: 
                        healthy_samples.append((file, i, 0))
                else:
                    healthy_samples.append((file, i, 0))
        
        # Perfect 50/50 split for training to force tumor detection
        if self.is_training:
            np.random.shuffle(healthy_samples)
            healthy_samples = healthy_samples[:len(tumor_samples)]
            
        all_samples = tumor_samples + healthy_samples
        np.random.shuffle(all_samples)
        return all_samples

    def __len__(self):
        return int(np.ceil(len(self.samples) / BATCH_SIZE))

    def __getitem__(self, idx):
        batch = self.samples[idx * BATCH_SIZE : (idx + 1) * BATCH_SIZE]
        X, y = [], []
        for file, slice_idx, label in batch:
            if file not in self.cache:
                self.cache[file] = nib.load(os.path.join(image_path, file)).get_fdata()
            
            # --- WINDOWING FOR SOFT TISSUE (Critical for Early Stage) ---
            slice_img = np.clip(self.cache[file][:, :, slice_idx], -100, 200)
            slice_img = (slice_img + 100) / 300.0
            slice_img = cv2.resize(slice_img, (IMG_SIZE, IMG_SIZE))
            # EfficientNet expects [0, 255] then uses internal scaling, or [0, 1]
            X.append(cv2.cvtColor((slice_img * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB))
            y.append(label)
        
        if len(self.cache) > 3: self.cache = {} 
        return np.array(X), np.array(y)

# --- MODEL: EfficientNet-B0 ---
# Better for small tumors (< 2cm) than MobileNet
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = True # Unfreeze for early stage texture learning

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(learning_rate=1e-5), loss=focal_loss(), metrics=['accuracy', tf.keras.metrics.Recall()])

# --- TRAINING ---
train_gen = PancreasGenerator(train_files, is_training=True)
model.fit(train_gen, epochs=5)

# --- PATIENT-LEVEL EVALUATION ---
print("\nPerforming Patient-Level Diagnosis...")
y_true_patient = []
y_pred_patient = []

for file in test_files:
    img_vol = nib.load(os.path.join(image_path, file)).get_fdata()
    mask_vol = nib.load(os.path.join(mask_path, file)).get_fdata()
    
    y_true_patient.append(1 if np.any(mask_vol == 2) else 0)
    
    slice_scores = []
    # Scan patient with high resolution (every 2nd slice) to not miss small tumors
    for i in range(0, img_vol.shape[2], 2):
        img = np.clip(img_vol[:,:,i], -100, 200)
        img = cv2.resize((img + 100) / 300.0, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
        score = model.predict(np.expand_dims(img, axis=0), verbose=0)[0][0]
        slice_scores.append(score)
    
    # Early detection logic: If any slice shows high probability, patient is Positive
    y_pred_patient.append(1 if np.max(slice_scores) > 0.5 else 0)

print("\nFINAL PATIENT-LEVEL CLASSIFICATION REPORT:")
print(classification_report(y_true_patient, y_pred_patient))

2026-03-24 10:53:21.164592: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774349601.350657      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774349601.406501      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774349601.838607      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774349601.838650      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774349601.838653      55 computation_placer.cc:177] computation placer alr

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5


I0000 00:00:1774349763.776537     122 service.cc:152] XLA service 0x7a332c12f620 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774349763.776582     122 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1774349770.561065     122 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-24 10:56:21.714945: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-24 10:56:21.899829: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-24 10:56:22.328461: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accur

226/259 ━━━━━━━━━━━━━━━━━━━━ 1:10 2s/step - accuracy: 0.5180 - loss: 0.2880 - recall: 0.5080

2026-03-24 11:04:57.194298: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-24 11:04:57.378950: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-24 11:04:57.776631: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-24 11:04:57.981424: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


259/259 ━━━━━━━━━━━━━━━━━━━━ 646s 2s/step - accuracy: 0.5244 - loss: 0.2824 - recall: 0.5149
Epoch 2/5
259/259 ━━━━━━━━━━━━━━━━━━━━ 352s 1s/step - accuracy: 0.6496 - loss: 0.1705 - recall: 0.6196
Epoch 3/5
259/259 ━━━━━━━━━━━━━━━━━━━━ 349s 1s/step - accuracy: 0.6880 - loss: 0.1445 - recall: 0.6531
Epoch 4/5
259/259 ━━━━━━━━━━━━━━━━━━━━ 349s 1s/step - accuracy: 0.7206 - loss: 0.1262 - recall: 0.6888
Epoch 5/5
259/259 ━━━━━━━━━━━━━━━━━━━━ 347s 1s/step - accuracy: 0.7373 - loss: 0.1214 - recall: 0.7088

Performing Patient-Level Diagnosis...

FINAL PATIENT-LEVEL CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        57

    accuracy                           1.00        57
   macro avg       1.00      1.00      1.00        57
weighted avg       1.00      1.00      1.00        57

